In [1]:
#!/usr/bin/env python
# coding: utf-8


# TS-SatFire AF Detection -- SE-UNet3D Inference

Evaluates our best model on the 15 test fires with verified AF labels.
Produces per-fire metrics, TP/FP/FN maps, threshold sweep, and
paper-style comparison figures.

**Requires:** training checkpoint (.pt) and ts-satfire dataset as Kaggle inputs.


In [2]:
import os, sys, time, glob, random, warnings, json
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast
from tqdm.auto import tqdm
import rasterio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | Device: {DEVICE}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} -- {p.total_memory/1e9:.1f} GB")


PyTorch: 2.10.0+cu128 | CUDA: 12.8 | Device: cuda
  GPU 0: Tesla T4 -- 15.6 GB
  GPU 1: Tesla T4 -- 15.6 GB


## Cell 2: Auto-detect Files

Searches for the checkpoint (.pt) and ts-satfire dataset automatically.
Excludes the ts-satfire dataset folder from the weight search for speed.


In [3]:
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "fire_maps"), exist_ok=True)

# ---- Find ts-satfire dataset ----
DATA_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "ts-satfire" in dirs:
        candidate = os.path.join(root, "ts-satfire")
        # Check for nested ts-satfire/ts-satfire
        inner = os.path.join(candidate, "ts-satfire")
        if os.path.isdir(inner):
            # Verify it has fire directories
            items = os.listdir(inner)
            if any(d.isdigit() for d in items):
                DATA_ROOT = inner
                break
        else:
            items = os.listdir(candidate)
            if any(d.isdigit() for d in items):
                DATA_ROOT = candidate
                break
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth >= 4:
        dirs.clear()

if DATA_ROOT is None:
    # Fallback: brute force search for a directory with numeric fire IDs
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(d.isdigit() and len(d) >= 7 for d in dirs):
            if any(d.startswith("elephant") or d.startswith("dixie") for d in dirs):
                DATA_ROOT = root
                break
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth >= 5:
            dirs.clear()

print(f"Dataset: {DATA_ROOT}")
assert DATA_ROOT and os.path.isdir(DATA_ROOT), "ts-satfire dataset not found!"

# ---- Find checkpoint (.pt file) -- skip ts-satfire folder ----
CKPT_PATH = None
SATFIRE_PARENT = os.path.dirname(DATA_ROOT)  # skip this tree

def find_checkpoint():
    """Search for .pt files outside the ts-satfire dataset."""
    candidates = []
    for root, dirs, files in os.walk("/kaggle/input"):
        # Skip the ts-satfire dataset tree
        if root.startswith(SATFIRE_PARENT) and "ts-satfire" in root:
            dirs.clear()
            continue
        for f in files:
            if f.endswith(".pt"):
                full = os.path.join(root, f)
                size_mb = os.path.getsize(full) / 1e6
                candidates.append((full, size_mb))
                print(f"  Found: {full} ({size_mb:.1f} MB)")
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth >= 5:
            dirs.clear()

    # Also check /kaggle/working
    for f in glob.glob("/kaggle/working/**/*.pt", recursive=True):
        size_mb = os.path.getsize(f) / 1e6
        candidates.append((f, size_mb))
        print(f"  Found: {f} ({size_mb:.1f} MB)")

    if not candidates:
        return None
    # Pick the largest .pt file (most likely the model checkpoint)
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]

print("\nSearching for model checkpoint...")
CKPT_PATH = find_checkpoint()
print(f"\nUsing checkpoint: {CKPT_PATH}")
assert CKPT_PATH, "No .pt checkpoint found! Add training output as Kaggle dataset."


Dataset: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire

Searching for model checkpoint...
  Found: /kaggle/input/datasets/con1los/af-v6-day1/best_v6.pt (131.6 MB)

Using checkpoint: /kaggle/input/datasets/con1los/af-v6-day1/best_v6.pt


## Cell 3: Configuration

Must match training exactly: 8 channels, TS=2, 256x256 crop.


In [4]:
SEED = 42
TS_LENGTH = 1
N_CHANNELS = 8
IMAGE_SIZE = 256
THRESHOLD = 0.5  # will be optimized later

MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                 294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

# 15 test fires with verified labels (exclude calfcanyon_fire, mosquito_fire)
AF_TEST_FIRES = [
    "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire",
    "sparks_lake_fire", "lytton_fire", "chuckegg_creek_fire",
    "swedish_fire", "sydney_fire", "thomas_fire", "tubbs_fire",
    "carr_fire", "camp_fire", "creek_fire", "blue_ridge_fire",
    "dixie_fire",
]
EXCLUDED_TEST = ["calfcanyon_fire", "mosquito_fire"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"Test fires: {len(AF_TEST_FIRES)} (excluded {len(EXCLUDED_TEST)} with no labels)")
print(f"TS={TS_LENGTH}, patch={IMAGE_SIZE}, channels={N_CHANNELS}")


Test fires: 15 (excluded 2 with no labels)
TS=1, patch=256, channels=8


## Cell 4: Model Architecture

Identical SE-UNet3D from training. Must match exactly for weight loading.


In [5]:
class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch//r, bias=False), nn.ReLU(True),
                                nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)

class ResBlock3D(nn.Module):
    def __init__(self, ic, oc, r=8, dr=0.1):
        super().__init__()
        self.c1 = nn.Conv3d(ic, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b1 = nn.BatchNorm3d(oc)
        self.c2 = nn.Conv3d(oc, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, r)
        self.relu = nn.ReLU(True)
        self.drop = nn.Dropout3d(dr) if dr > 0 else nn.Identity()
        self.skip = (nn.Sequential(nn.Conv3d(ic, oc, 1, bias=False),
                     nn.BatchNorm3d(oc)) if ic != oc else nn.Identity())
    def forward(self, x):
        r = self.skip(x)
        o = self.relu(self.b1(self.c1(x)))
        o = self.drop(o)
        o = self.b2(self.c2(o))
        o = self.se(o)
        return self.relu(o + r)

class SEUNet3D(nn.Module):
    def __init__(self, ic=8, nc=1, ec=(64,128,256,512), r=8, dr=0.1):
        super().__init__()
        self.e1 = ResBlock3D(ic, ec[0], r, dr)
        self.e2 = ResBlock3D(ec[0], ec[1], r, dr)
        self.e3 = ResBlock3D(ec[1], ec[2], r, dr)
        self.e4 = ResBlock3D(ec[2], ec[3], r, dr)
        self.pool = nn.MaxPool3d((1,2,2), stride=(1,2,2))
        self.bot = ResBlock3D(ec[3], ec[3]*2, r, dr)
        self.u4 = nn.ConvTranspose3d(ec[3]*2, ec[3], (1,2,2), stride=(1,2,2))
        self.d4 = ResBlock3D(ec[3]*2, ec[3], r, dr)
        self.u3 = nn.ConvTranspose3d(ec[3], ec[2], (1,2,2), stride=(1,2,2))
        self.d3 = ResBlock3D(ec[2]*2, ec[2], r, dr)
        self.u2 = nn.ConvTranspose3d(ec[2], ec[1], (1,2,2), stride=(1,2,2))
        self.d2 = ResBlock3D(ec[1]*2, ec[1], r, dr)
        self.u1 = nn.ConvTranspose3d(ec[1], ec[0], (1,2,2), stride=(1,2,2))
        self.d1 = ResBlock3D(ec[0]*2, ec[0], r, dr)
        self.final = nn.Conv3d(ec[0], nc, 1)
        self.ds3 = nn.Conv3d(ec[2], nc, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.bot(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.final(d1)

# Load model
model = SEUNet3D(ic=N_CHANNELS, nc=1, ec=(64,128,256,512), r=8, dr=0.1).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
state = ckpt["model_state_dict"]
# Handle DataParallel prefix
if any(k.startswith("module.") for k in state):
    state = {k.replace("module.", ""): v for k, v in state.items()}
model.load_state_dict(state, strict=False)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
train_f1 = ckpt.get("f1", "N/A")
train_epoch = ckpt.get("epoch", "N/A")
print(f"Model loaded: {n_params/1e6:.2f}M params")
print(f"Checkpoint: epoch={train_epoch}, val_F1={train_f1}")


Model loaded: 32.88M params
Checkpoint: epoch=55, val_F1=0.8225687644850075


## Cell 5: Data Loading Utilities


In [6]:
def load_frame(fire_dir, day_path, return_label=False):
    """Load 8-channel frame + optional AF label from band 7."""
    with rasterio.open(day_path) as src:
        day_arr = src.read().astype(np.float32)
    day_bands = day_arr[:6]
    H, W = day_bands.shape[1], day_bands.shape[2]
    label = None
    if return_label and day_arr.shape[0] >= 7:
        b7 = day_arr[6]
        if np.isnan(b7).sum() < b7.size:
            label = (b7 >= 7).astype(np.float32)

    night_dir = os.path.join(fire_dir, "VIIRS_Night")
    night_path = os.path.join(night_dir,
        os.path.basename(day_path).replace("_VIIRS_Day", "_VIIRS_Night"))
    if os.path.exists(night_path):
        with rasterio.open(night_path) as src:
            na = src.read().astype(np.float32)
        nb = na[:2, :H, :W] if na.shape[0] >= 2 else np.zeros((2, H, W), dtype=np.float32)
    else:
        nb = np.zeros((2, H, W), dtype=np.float32)
    frame = np.concatenate([day_bands, nb], axis=0)
    return (frame, label) if return_label else frame


def prepare_window(fire_dir, day_files, t_start, ts_length, mean, std, patch_size):
    """Load a T-length window, normalize, center crop. Returns (1,C,T,H,W) tensor + label."""
    frames, label = [], None
    H = W = None
    for t in range(t_start, t_start + ts_length):
        is_last = (t == t_start + ts_length - 1)
        if is_last:
            fr, label = load_frame(fire_dir, day_files[t], return_label=True)
        else:
            fr = load_frame(fire_dir, day_files[t])
        if H is None:
            H, W = fr.shape[1], fr.shape[2]
        frames.append(fr[:, :H, :W])

    if label is not None:
        label = label[:H, :W]

    stack = np.stack(frames, axis=0)  # (T, 8, H, W)
    stack = (stack - mean[None, :, None, None]) / (std[None, :, None, None] + 1e-8)
    stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)

    # Center crop
    r0 = (H - patch_size) // 2; c0 = (W - patch_size) // 2
    stack = stack[:, :, r0:r0+patch_size, c0:c0+patch_size]
    if label is not None:
        label = label[r0:r0+patch_size, c0:c0+patch_size]

    x = torch.from_numpy(stack.transpose(1, 0, 2, 3)).float().unsqueeze(0)  # (1,C,T,H,W)
    return x, label, frames[-1]  # also return raw last frame for visualization

print("Data utilities ready.")


Data utilities ready.


## Cell 6: Per-Fire Evaluation

For each test fire, run sliding-window inference and collect
TP/FP/FN pixel counts. Skip windows where label is None (NaN days).


In [7]:
@torch.no_grad()
def evaluate_fire(fire_id, threshold=0.5):
    """Run inference on one fire, return metrics + predictions for vis."""
    fdir = os.path.join(DATA_ROOT, fire_id)
    day_files = sorted(glob.glob(os.path.join(fdir, "VIIRS_Day", "*.tif")))

    if len(day_files) < TS_LENGTH:
        return None

    tp_total = fp_total = fn_total = 0
    n_windows = 0
    n_skipped = 0
    all_probs = []
    all_labels = []
    vis_data = []  # for qualitative plots

    for t0 in range(len(day_files) - TS_LENGTH + 1):
        x, label, raw_frame = prepare_window(
            fdir, day_files, t0, TS_LENGTH, MEAN, STD, IMAGE_SIZE)

        if label is None:
            n_skipped += 1
            continue

        x = x.to(DEVICE)
        with autocast(enabled=True):
            logits = model(x)
        probs = torch.sigmoid(logits[:, 0, -1]).cpu().numpy()[0]  # (H, W)
        pred = (probs > threshold).astype(np.float32)
        lbl = label.astype(np.float32)

        tp = int(((pred == 1) & (lbl == 1)).sum())
        fp = int(((pred == 1) & (lbl == 0)).sum())
        fn = int(((pred == 0) & (lbl == 1)).sum())

        tp_total += tp; fp_total += fp; fn_total += fn
        n_windows += 1
        all_probs.append(probs.flatten())
        all_labels.append(lbl.flatten())

        # Save last window for visualization
        vis_data.append({
            "raw": raw_frame, "label": lbl, "pred": pred, "probs": probs,
            "date": os.path.basename(day_files[t0 + TS_LENGTH - 1]).replace("_VIIRS_Day.tif", "")
        })

    if n_windows == 0:
        return None

    prec = tp_total / max(tp_total + fp_total, 1)
    rec = tp_total / max(tp_total + fn_total, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    iou = tp_total / max(tp_total + fp_total + fn_total, 1)

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    return {
        "fire_id": fire_id, "n_windows": n_windows, "n_skipped": n_skipped,
        "tp": tp_total, "fp": fp_total, "fn": fn_total,
        "f1": f1, "iou": iou, "precision": prec, "recall": rec,
        "probs": all_probs, "labels": all_labels,
        "vis": vis_data,
    }


# Run evaluation on all test fires
print(f"Evaluating {len(AF_TEST_FIRES)} test fires (threshold={THRESHOLD})...")
print(f"{'Fire':<25s} {'Win':>4} {'Skip':>4} {'TP':>6} {'FP':>6} {'FN':>6} "
      f"{'F1':>7} {'IoU':>7} {'P':>6} {'R':>6}")
print("-" * 90)

all_results = []
all_test_probs = []
all_test_labels = []

for fid in tqdm(AF_TEST_FIRES, desc="Test fires", ncols=80):
    result = evaluate_fire(fid, THRESHOLD)
    if result is None:
        print(f"{fid:<25s}  SKIPPED (no valid windows)")
        continue
    all_results.append(result)
    all_test_probs.append(result["probs"])
    all_test_labels.append(result["labels"])

    print(f"{fid:<25s} {result['n_windows']:>4} {result['n_skipped']:>4} "
          f"{result['tp']:>6} {result['fp']:>6} {result['fn']:>6} "
          f"{result['f1']:>7.4f} {result['iou']:>7.4f} "
          f"{result['precision']:>6.3f} {result['recall']:>6.3f}")

# Aggregate metrics
total_tp = sum(r["tp"] for r in all_results)
total_fp = sum(r["fp"] for r in all_results)
total_fn = sum(r["fn"] for r in all_results)
agg_prec = total_tp / max(total_tp + total_fp, 1)
agg_rec = total_tp / max(total_tp + total_fn, 1)
agg_f1 = 2 * agg_prec * agg_rec / max(agg_prec + agg_rec, 1e-8)
agg_iou = total_tp / max(total_tp + total_fp + total_fn, 1)
mean_f1 = np.mean([r["f1"] for r in all_results])

print(f"\n{'='*90}")
print(f"AGGREGATE TEST METRICS ({len(all_results)} fires, threshold={THRESHOLD})")
print(f"{'='*90}")
print(f"  F1 (micro):   {agg_f1:.4f}")
print(f"  IoU (micro):  {agg_iou:.4f}")
print(f"  Precision:    {agg_prec:.4f}")
print(f"  Recall:       {agg_rec:.4f}")
print(f"  mF1 (macro):  {mean_f1:.4f}")
print(f"  Paper SwinUNETR-3D (TS=2): F1=0.823")
print(f"  Delta: {agg_f1 - 0.823:+.4f}")


Evaluating 15 test fires (threshold=0.5)...
Fire                       Win Skip     TP     FP     FN      F1     IoU      P      R
------------------------------------------------------------------------------------------


Test fires:   0%|                                        | 0/15 [00:00<?, ?it/s]

elephant_hill_fire          10    0   5389   1552   1044  0.8059  0.6749  0.776  0.838
eagle_bluff_fire             5    5    519     97     77  0.8564  0.7489  0.843  0.871
double_creek_fire            3    7    237     27     58  0.8479  0.7360  0.898  0.803
sparks_lake_fire             9    1   5957   1301    756  0.8528  0.7433  0.821  0.887
lytton_fire                 10    0   1361    508    238  0.7849  0.6459  0.728  0.851
chuckegg_creek_fire          8    2  15873   3142   3719  0.8223  0.6982  0.835  0.810
swedish_fire                 9    1    846    170    111  0.8576  0.7507  0.833  0.884
sydney_fire                 10    0   8284   2020   1465  0.8262  0.7039  0.804  0.850
thomas_fire                 10    0  10721   1572   1827  0.8632  0.7593  0.872  0.854
tubbs_fire                  10    0   7648   1125   1539  0.8517  0.7417  0.872  0.832
carr_fire                   10    0   7058    770   1308  0.8717  0.7725  0.902  0.844
camp_fire                   10    0   6953 

## Cell 7: Threshold Sweep on Test Set


In [8]:
all_probs_flat = np.concatenate(all_test_probs)
all_labels_flat = np.concatenate(all_test_labels)

thresholds = np.arange(0.20, 0.81, 0.02)
sweep = []
for thr in thresholds:
    p = (all_probs_flat > thr).astype(float)
    tp = int(((p == 1) & (all_labels_flat == 1)).sum())
    fp = int(((p == 1) & (all_labels_flat == 0)).sum())
    fn = int(((p == 0) & (all_labels_flat == 1)).sum())
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2*prec*rec / max(prec+rec, 1e-8)
    iou = tp / max(tp+fp+fn, 1)
    sweep.append({"thr": thr, "f1": f1, "iou": iou, "prec": prec, "rec": rec})

sweep_df = pd.DataFrame(sweep)
best_row = sweep_df.loc[sweep_df["f1"].idxmax()]
opt_thr = float(best_row["thr"])
opt_f1 = float(best_row["f1"])
opt_iou = float(best_row["iou"])

print(f"{'Thr':>5} {'F1':>7} {'IoU':>7} {'P':>6} {'R':>6}")
print("-" * 38)
for _, r in sweep_df.iterrows():
    m = " <<<" if r["thr"] == opt_thr else ""
    print(f"{r['thr']:5.2f} {r['f1']:7.4f} {r['iou']:7.4f} "
          f"{r['prec']:6.3f} {r['rec']:6.3f}{m}")

print(f"\nBest threshold: {opt_thr:.2f} -> F1={opt_f1:.4f} "
      f"(default 0.50: {agg_f1:.4f}, gain: {opt_f1-agg_f1:+.4f})")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep_df["thr"], sweep_df["f1"], "g-o", lw=2, ms=4, label="F1")
ax.plot(sweep_df["thr"], sweep_df["iou"], "m-s", lw=1.5, ms=3, label="IoU")
ax.plot(sweep_df["thr"], sweep_df["prec"], "c--", lw=1, label="Precision")
ax.plot(sweep_df["thr"], sweep_df["rec"], "y--", lw=1, label="Recall")
ax.axvline(opt_thr, color="red", ls=":", label=f"Optimal ({opt_thr:.2f})")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title("Test Set Threshold Sweep"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "test_threshold_sweep.png"), dpi=150)
plt.show(); plt.close()


  Thr      F1     IoU      P      R
--------------------------------------
 0.20  0.8538  0.7449  0.841  0.867
 0.22  0.8538  0.7450  0.842  0.866 <<<
 0.24  0.8538  0.7449  0.843  0.865
 0.26  0.8537  0.7448  0.844  0.864
 0.28  0.8537  0.7447  0.845  0.862
 0.30  0.8537  0.7447  0.846  0.862
 0.32  0.8536  0.7446  0.847  0.861
 0.34  0.8535  0.7445  0.847  0.860
 0.36  0.8534  0.7444  0.848  0.859
 0.38  0.8535  0.7444  0.849  0.858
 0.40  0.8534  0.7444  0.849  0.857
 0.42  0.8534  0.7443  0.850  0.857
 0.44  0.8532  0.7440  0.851  0.856
 0.46  0.8531  0.7439  0.851  0.855
 0.48  0.8531  0.7438  0.852  0.854
 0.50  0.8530  0.7437  0.852  0.854
 0.52  0.8530  0.7437  0.853  0.853
 0.54  0.8528  0.7434  0.853  0.852
 0.56  0.8527  0.7433  0.854  0.851
 0.58  0.8525  0.7429  0.855  0.851
 0.60  0.8524  0.7428  0.855  0.850
 0.62  0.8523  0.7427  0.856  0.849
 0.64  0.8523  0.7426  0.857  0.848
 0.66  0.8522  0.7425  0.857  0.847
 0.68  0.8520  0.7422  0.858  0.846
 0.70  0.8520  0.7421

## Cell 8: Per-Fire TP/FP/FN Maps (Paper-Style)

For each test fire, plot Band I4 with colored overlay:
- **Red**: True Positive (correctly detected fire)
- **Green**: False Positive (predicted fire where there is none)
- **Blue**: False Negative (missed fire)


In [9]:
# ---- Fixed Cell 8: Paper-style TP/FP/FN Maps ----
# Red = True Positive, Green = False Positive, Blue = False Negative
# Uses solid RGBA overlays matching the paper's color scheme

from matplotlib.colors import ListedColormap

n_fires = len(all_results)
cols = 3
rows = (n_fires + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
if rows == 1:
    axes = axes.reshape(1, -1)

for idx, result in enumerate(sorted(all_results, key=lambda x: x["f1"], reverse=True)):
    row, col = idx // cols, idx % cols
    ax = axes[row, col]

    vis = result["vis"][-1]
    raw = vis["raw"]
    lbl = vis["label"]
    pred = vis["pred"]

    # Band I4 for background
    i4 = raw[3]
    i4_disp = np.where(np.isfinite(i4), i4, np.nanmin(i4[np.isfinite(i4)]))
    i4_norm = (i4_disp - i4_disp.min()) / (i4_disp.max() - i4_disp.min() + 1e-8)

    H, W = raw.shape[1], raw.shape[2]
    r0 = (H - IMAGE_SIZE) // 2; c0 = (W - IMAGE_SIZE) // 2
    i4_crop = i4_norm[r0:r0+IMAGE_SIZE, c0:c0+IMAGE_SIZE]

    ax.imshow(i4_crop, cmap="gray", vmin=0, vmax=1)

    # Build RGBA overlay: transparent background, solid colors for TP/FP/FN
    overlay = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 4), dtype=np.float32)

    tp_mask = (pred == 1) & (lbl == 1)
    fp_mask = (pred == 1) & (lbl == 0)
    fn_mask = (pred == 0) & (lbl == 1)

    # Red for TP (R=1, G=0, B=0, A=0.7)
    overlay[tp_mask] = [1.0, 0.0, 0.0, 0.7]
    # Green for FP (R=0, G=0.8, B=0, A=0.7)
    overlay[fp_mask] = [0.0, 0.8, 0.0, 0.7]
    # Blue for FN (R=0, G=0, B=1, A=0.7)
    overlay[fn_mask] = [0.0, 0.0, 1.0, 0.7]

    ax.imshow(overlay)

    fid = result["fire_id"].replace("_fire", "").replace("_", " ").title()
    ax.set_title(f"{fid}\nF1={result['f1']:.3f} P={result['precision']:.2f} R={result['recall']:.2f}",
                 fontsize=9)
    ax.axis("off")

# Remove empty subplots
for idx in range(n_fires, rows * cols):
    row, col = idx // cols, idx % cols
    axes[row, col].axis("off")

# Legend with matching colors
tp_p = mpatches.Patch(color=(1.0, 0.0, 0.0), label="True Positive")
fp_p = mpatches.Patch(color=(0.0, 0.8, 0.0), label="False Positive")
fn_p = mpatches.Patch(color=(0.0, 0.0, 1.0), label="False Negative")
fig.legend(handles=[tp_p, fp_p, fn_p], loc="lower center", ncol=3, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle(f"Active Fire Detection -- SE-UNet3D (TS=2)\n"
             f"Test F1={agg_f1:.4f} | Optimal F1={opt_f1:.4f} (thr={opt_thr:.2f})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
fig.subplots_adjust(top=0.90)
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "all_fires_tpfpfn.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: plots/all_fires_tpfpfn.png")

Saved: plots/all_fires_tpfpfn.png


## Cell 9: Per-Fire Bar Chart


In [10]:
fire_df = pd.DataFrame([{
    "fire_id": r["fire_id"].replace("_fire", "").replace("_", " ").title(),
    "f1": r["f1"], "iou": r["iou"], "precision": r["precision"], "recall": r["recall"],
    "tp": r["tp"], "fp": r["fp"], "fn": r["fn"],
} for r in all_results]).sort_values("f1", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#e74c3c" if f < 0.5 else "#f39c12" if f < 0.75 else "#2ecc71"
          for f in fire_df["f1"]]
ax.barh(fire_df["fire_id"], fire_df["f1"], color=colors, edgecolor="gray", alpha=0.85)
ax.axvline(agg_f1, color="navy", ls="--", lw=2, label=f"Aggregate F1={agg_f1:.3f}")
ax.axvline(0.823, color="red", ls=":", lw=1.5, label="Paper SwinUNETR-3D (0.823)")
ax.set_xlabel("F1 Score"); ax.set_title("Per-Fire F1 Scores", fontweight="bold")
ax.legend(); ax.grid(axis="x", alpha=0.3)
for i, (_, r) in enumerate(fire_df.iterrows()):
    ax.text(r["f1"] + 0.005, i, f"{r['f1']:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_fire_f1.png"), dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: plots/per_fire_f1.png")


Saved: plots/per_fire_f1.png


## Cell 10: Model Comparison


In [11]:
models_cmp = OrderedDict([
    ("U-Net (2D)",            (0.731, 0.605)),
    ("Att-UNet (2D)",         (0.763, 0.648)),
    ("UNETR-2D",              (0.733, 0.621)),
    ("SwinUNETR-2D",          (0.774, 0.660)),
    ("GRU-3",                 (0.713, 0.601)),
    ("LSTM-3",                (0.765, 0.654)),
    ("T4Fire",                (0.802, 0.700)),
    ("U-Net-3D",              (0.748, 0.628)),
    ("Att-UNet-3D",           (0.770, 0.654)),
    ("UNETR-3D",              (0.811, 0.706)),
    ("SwinUNETR-3D (TS=6)",   (0.797, 0.688)),
    ("SwinUNETR-3D (TS=2)",   (0.823, 0.727)),
    ("Ours (SE-UNet3D, TS=2)",(opt_f1, opt_iou)),
])
names = list(models_cmp.keys())
f1s = [v[0] for v in models_cmp.values()]
colors = ["#6baed6"] * (len(names)-1) + ["#e6550d"]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(names, f1s, color=colors, edgecolor="gray", alpha=0.85)
ax.set_xlabel("F1 Score")
ax.set_title("F1 Comparison -- AF Detection (TS=2)", fontweight="bold")
ax.grid(axis="x", alpha=0.3)
for i, v in enumerate(f1s):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "model_comparison_test.png"),
            dpi=150, bbox_inches="tight")
plt.show(); plt.close()

print(f"\n{'Model':<30s} {'F1':>7} {'IoU':>7}")
print("-" * 48)
for n, (f, io) in models_cmp.items():
    m = " <<<" if "Ours" in n else ""
    print(f"{n:<30s} {f:7.3f} {io:7.3f}{m}")



Model                               F1     IoU
------------------------------------------------
U-Net (2D)                       0.731   0.605
Att-UNet (2D)                    0.763   0.648
UNETR-2D                         0.733   0.621
SwinUNETR-2D                     0.774   0.660
GRU-3                            0.713   0.601
LSTM-3                           0.765   0.654
T4Fire                           0.802   0.700
U-Net-3D                         0.748   0.628
Att-UNet-3D                      0.770   0.654
UNETR-3D                         0.811   0.706
SwinUNETR-3D (TS=6)              0.797   0.688
SwinUNETR-3D (TS=2)              0.823   0.727
Ours (SE-UNet3D, TS=2)           0.854   0.745 <<<


## Cell 11: Save All Results


In [12]:
# Per-fire CSV
fire_results_df = pd.DataFrame([{
    "fire_id": r["fire_id"], "n_windows": r["n_windows"],
    "tp": r["tp"], "fp": r["fp"], "fn": r["fn"],
    "f1": round(r["f1"], 4), "iou": round(r["iou"], 4),
    "precision": round(r["precision"], 4), "recall": round(r["recall"], 4),
} for r in all_results])
fire_results_df.to_csv(os.path.join(OUTPUT_DIR, "per_fire_test_results.csv"), index=False)

# JSON summary
results = {
    "model": "SE-UNet3D",
    "n_params_M": round(n_params / 1e6, 2),
    "ts_length": TS_LENGTH,
    "checkpoint": os.path.basename(CKPT_PATH),
    "train_val_f1": float(train_f1) if isinstance(train_f1, (int, float)) else str(train_f1),
    "n_test_fires": len(all_results),
    "excluded_fires": EXCLUDED_TEST,
    "default_threshold": 0.5,
    "optimal_threshold": round(opt_thr, 2),
    "test_f1_default": round(agg_f1, 4),
    "test_f1_optimal": round(opt_f1, 4),
    "test_iou_optimal": round(opt_iou, 4),
    "test_precision": round(float(best_row["prec"]), 4),
    "test_recall": round(float(best_row["rec"]), 4),
    "test_mf1": round(mean_f1, 4),
    "paper_f1": 0.823,
    "paper_iou": 0.727,
    "delta_f1": round(opt_f1 - 0.823, 4),
    "per_fire": [{
        "id": r["fire_id"], "f1": round(r["f1"], 4),
        "iou": round(r["iou"], 4)
    } for r in all_results],
}

with open(os.path.join(OUTPUT_DIR, "test_results.json"), "w") as f:
    json.dump(results, f, indent=2)

# Print final summary
print(json.dumps(results, indent=2))

print(f"\n{'='*70}")
print(f"FINAL TEST RESULTS -- SE-UNet3D (TS=2)")
print(f"{'='*70}")
print(f"  Test fires:      {len(all_results)}/17 (excluded {len(EXCLUDED_TEST)} with no labels)")
print(f"  F1 (thr=0.50):   {agg_f1:.4f}")
print(f"  F1 (thr={opt_thr:.2f}):   {opt_f1:.4f}")
print(f"  IoU:              {opt_iou:.4f}")
print(f"  Paper SwinUNETR:  F1=0.823, IoU=0.727")
print(f"  Delta:            F1={opt_f1-0.823:+.4f}, IoU={opt_iou-0.727:+.4f}")
if opt_f1 > 0.823:
    print(f"\n  >>> BEATS PAPER BASELINE ON TEST SET <<<")
print(f"\n  Output files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        print(f"    {f} ({os.path.getsize(fp)/1024:.0f} KB)")
    elif os.path.isdir(fp):
        print(f"    {f}/ ({len(os.listdir(fp))} files)")
print(f"{'='*70}")


{
  "model": "SE-UNet3D",
  "n_params_M": 32.88,
  "ts_length": 1,
  "checkpoint": "best_v6.pt",
  "train_val_f1": 0.8225687644850075,
  "n_test_fires": 15,
  "excluded_fires": [
    "calfcanyon_fire",
    "mosquito_fire"
  ],
  "default_threshold": 0.5,
  "optimal_threshold": 0.22,
  "test_f1_default": 0.853,
  "test_f1_optimal": 0.8538,
  "test_iou_optimal": 0.745,
  "test_precision": 0.8423,
  "test_recall": 0.8657,
  "test_mf1": 0.8425,
  "paper_f1": 0.823,
  "paper_iou": 0.727,
  "delta_f1": 0.0308,
  "per_fire": [
    {
      "id": "elephant_hill_fire",
      "f1": 0.8059,
      "iou": 0.6749
    },
    {
      "id": "eagle_bluff_fire",
      "f1": 0.8564,
      "iou": 0.7489
    },
    {
      "id": "double_creek_fire",
      "f1": 0.8479,
      "iou": 0.736
    },
    {
      "id": "sparks_lake_fire",
      "f1": 0.8528,
      "iou": 0.7433
    },
    {
      "id": "lytton_fire",
      "f1": 0.7849,
      "iou": 0.6459
    },
    {
      "id": "chuckegg_creek_fire",
      "f1":